In [1]:
import ROOT
# print(f"ROOT version: {ROOT.__version__}")
import os 
import time 
import numpy as np
from scipy.special import j0
import plotly.graph_objects as go
from scipy.integrate import fixed_quad


In [2]:
save_folder = 'run8'
n_points = 10000

n=3

lower_factor = 0.99
upper_factor = 2 - lower_factor

b_max = 30
q_max = 0.2



In [3]:
def read_data_file(filename):
    """Read data file and return arrays for x, y, y_error"""
    x_vals = []
    y_vals = []
    y_errs = []
    
    with open(filename, 'r') as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                parts = line.split()
                if len(parts) >= 3:
                    x_vals.append(float(parts[0]))
                    y_vals.append(float(parts[1]))
                    y_errs.append(float(parts[2]))
    
    return x_vals, y_vals, y_errs

In [4]:
# Load experimental data - CHANGED to ROOT's TGraphErrors
x_atlas_all, y_atlas_all, yerr_atlas_all = read_data_file('../../../data/ens_atlas_difc0_2.dat')
x_totem_all, y_totem_all, yerr_totem_all = read_data_file('../../../data/ens_totem_difc0_2.dat')

# Function to process data for each experiment - CHANGED for ROOT objects
def process_data(x_data, y_data, yerr_data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        if end is None:
            end = len(x_data)
        x_values.append(x_data[start:end])
        y_values.append(y_data[start:end])
        y_errors.append(yerr_data[start:end])
    
    return x_values, y_values, y_errors


In [5]:
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data - CHANGED to use the lists
x_atlas, y_atlas, yerr_atlas = process_data(x_atlas_all, y_atlas_all, yerr_atlas_all, atlas_blocks)
x_totem, y_totem, yerr_totem = process_data(x_totem_all, y_totem_all, yerr_totem_all, totem_blocks)

# Extract values by energy - SAME
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [6]:
# print(x_7_atlas, y_7_atlas, yerr_7_atlas)
# print(x_8_atlas, y_8_atlas, yerr_8_atlas)
# print(x_13_atlas, y_13_atlas, yerr_13_atlas)

In [7]:
import math

b_0 = (33 - 6) / (12 * math.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25

ensemble_parameters = {
    'atlas': {
        'log': {
            'eps': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'eps': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    }
}

# Get parameters for selected configuration
initial_params_log_atlas = ensemble_parameters['atlas']['log']
initial_params_pl_atlas = ensemble_parameters['atlas']['pl']

In [8]:
import math

def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = math.log((q2 + rho_mg_squared) / lambda_squared) / math.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = math.log((q2 + rho_mg_squared) / lambda_squared) / math.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def G_p(q2, a1, a2):
    return math.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * math.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def born_sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def born_amp(diff_T, s, epsilon, t):
    
    alpha_pomeron = 1.0 + epsilon + 0.25 * t

    first_term = s**(alpha_pomeron)
    second_term = 1/(1**(alpha_pomeron-1))
    
    regge_factor = first_term * second_term
    return 1j * 8.0 * regge_factor * diff_T

In [9]:
import math
import cmath
import ROOT
from ROOT import Math

# -------------------------------
# Integral over phi
# -------------------------------
def phi_integral(k, mg, a1, a2, m2_func, q):
    def integrand(phi):
        return k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                    T_2(k, phi, mg, a1, a2, m2_func, q))
    
    # Create the functor
    func = ROOT.Math.Functor(integrand, 1)
    
    # Use kADAPTIVESINGULAR method - robust for potential singularities
    integ = ROOT.Math.IntegratorOneDim(ROOT.Math.IntegrationOneDim.kADAPTIVESINGULAR)
    
    # Set conservative tolerance settings
    integ.SetRelTolerance(1e-6)
    integ.SetAbsTolerance(1e-8)
    
    integ.SetFunction(func)
    return integ.Integral(0.0, 2*math.pi)
# -------------------------------
# Integral over k
# -------------------------------
def k_integral(k, mg, a1, a2, m2_func, q):
    return phi_integral(k, mg, a1, a2, m2_func, q)

# -------------------------------
# Double integral over phi and k
# -------------------------------
def compute_k_phi_integral(sqrt_s_val, mg, a1, a2, m2_func, q):
    def k_func(k):
        return k_integral(k, mg, a1, a2, m2_func, q)
    
    func = ROOT.Math.Functor(k_func, 1)
    
    # Use kADAPTIVESINGULAR method - robust for potential singularities
    integ = ROOT.Math.IntegratorOneDim(ROOT.Math.IntegrationOneDim.kADAPTIVESINGULAR)
    
    # Set conservative tolerance settings
    integ.SetRelTolerance(1e-6)
    integ.SetAbsTolerance(1e-8)
    
    integ.SetFunction(func)
    return integ.Integral(0.0, sqrt_s_val)

# -------------------------------
# Chi integral over q
# -------------------------------
def chi_integral(sqrt_s_val, b, q_max, born_amp_value):
    s = sqrt_s_val**2

    def q_integrand(q):
        return (q * j0(b * q) * born_amp_value) / s

    func_real = ROOT.Math.Functor(lambda q: q_integrand(q).real, 1)
    func_imag = ROOT.Math.Functor(lambda q: q_integrand(q).imag, 1)

    # Use kADAPTIVESINGULAR for both real and imaginary parts
    integ_real = ROOT.Math.IntegratorOneDim(ROOT.Math.IntegrationOneDim.kADAPTIVESINGULAR)
    integ_real.SetRelTolerance(1e-6)
    integ_real.SetAbsTolerance(1e-8)
    integ_real.SetFunction(func_real)
    real_part = integ_real.Integral(0.0, q_max)

    integ_imag = ROOT.Math.IntegratorOneDim(ROOT.Math.IntegrationOneDim.kADAPTIVESINGULAR)
    integ_imag.SetRelTolerance(1e-6)
    integ_imag.SetAbsTolerance(1e-8)
    integ_imag.SetFunction(func_imag)
    imag_part = integ_imag.Integral(0.0, q_max)

    return real_part + 1j * imag_part

# -------------------------------
# Eikonal amplitude integral over b
# -------------------------------
def eik_amp(sqrt_s_values, b_max, q_max, born_amp_value):
    if isinstance(sqrt_s_values, (int, float)):
        sqrt_s_values = [sqrt_s_values]

    amp_list = []
    for sqrt_s_val in sqrt_s_values:
        s = sqrt_s_val**2

        def b_integrand(b):
            chi_val = chi_integral(sqrt_s_val, b, q_max, born_amp_value)
            return b * (1 - cmath.exp(1j * chi_val))

        func_real = ROOT.Math.Functor(lambda b: b_integrand(b).real, 1)
        func_imag = ROOT.Math.Functor(lambda b: b_integrand(b).imag, 1)

        # Use kADAPTIVESINGULAR for both real and imaginary parts
        integ_real = ROOT.Math.IntegratorOneDim(ROOT.Math.IntegrationOneDim.kADAPTIVESINGULAR)
        integ_real.SetRelTolerance(1e-6)
        integ_real.SetAbsTolerance(1e-8)
        integ_real.SetFunction(func_real)
        real_part = integ_real.Integral(0.0, b_max)

        integ_imag = ROOT.Math.IntegratorOneDim(ROOT.Math.IntegrationOneDim.kADAPTIVESINGULAR)
        integ_imag.SetRelTolerance(1e-6)
        integ_imag.SetAbsTolerance(1e-8)
        integ_imag.SetFunction(func_imag)
        imag_part = integ_imag.Integral(0.0, b_max)

        A_eik = 1j * s * (real_part + 1j * imag_part)
        amp_list.append(A_eik)

    return amp_list if len(amp_list) > 1 else amp_list[0]

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag**2
    denominator = 16 * math.pi * s**2
    return amp_squared / denominator * 0.389379323


In [ ]:
import ROOT
import numpy as np

# escolha do modelo de massa
mass_model = "pl"
m2_func = m2_pl if mass_model == "pl" else m2_log

import ROOT
import numpy as np

# lista de valores de q para a integral eikonal (not needed for integration, just kept for reference)
lst_q_integration = np.linspace(0, 0.1, 130)

# escolha do modelo de massa
mass_model = "pl"
m2_func = m2_pl if mass_model == "pl" else m2_log

def model_function(x_born, eps, mg, a1, a2, sqrt_s, model_type="log", max_attempts=3):
    """
    x_born: array com os valores experimentais de q^2
    eps, mg, a1, a2: parâmetros do modelo
    sqrt_s: energia
    model_type: 'pl' ou 'log'
    max_attempts: número de tentativas por integração
    """
    m2_func = m2_log if model_type == "log" else m2_pl
    s = sqrt_s**2
    results = []

    def integrate_with_fallback(func, lower, upper, q_exp, point_index):
        """Helper function with robust integration strategy"""
        methods = [
            ('kADAPTIVESINGULAR', ROOT.Math.IntegrationOneDim.kADAPTIVESINGULAR),
            ('kADAPTIVE', ROOT.Math.IntegrationOneDim.kADAPTIVE),
            ('kNONADAPTIVE', ROOT.Math.IntegrationOneDim.kNONADAPTIVE)
        ]
        
        # Try different tolerance settings
        tolerance_settings = [
            (1e-4, 1e-6),  # Relaxed
            (1e-6, 1e-8),  # Default
            (1e-8, 1e-10)  # Strict
        ]
        
        for attempt, (method_name, method) in enumerate(methods):
            for tol_idx, (rel_tol, abs_tol) in enumerate(tolerance_settings):
                integ = ROOT.Math.IntegratorOneDim(method)
                integ.SetRelTolerance(rel_tol)
                integ.SetAbsTolerance(abs_tol)
                integ.SetFunction(func)
                
                try:
                    result = integ.Integral(lower, upper)
                    status = integ.Status()
                    
                    if status == 0:  # Success
                        return result, status
                    else:
                        print(f"q_exp={q_exp:.6f}: {method_name} tol={rel_tol:.0e} status={status}")
                        
                except Exception as e:
                    print(f"q_exp={q_exp:.6f}: {method_name} tol={rel_tol:.0e} exception: {e}")
        
        # If all methods fail, return a safe fallback
        print(f"CRITICAL: All integration methods failed for q_exp={q_exp:.6f}")
        return 0.0, -1

    for i, q_exp in enumerate(x_born):
        t = -q_exp

        def q_integrand(q):
            try:
                return compute_k_phi_integral(
                    sqrt_s_val=sqrt_s,  # Note: changed from s to sqrt_s to match your function signature
                    mg=mg,
                    a1=a1,
                    a2=a2,
                    m2_func=m2_func,
                    q=q
                )
            except Exception as e:
                print(f"Error in compute_k_phi_integral for q={q}: {e}")
                return 0.0

        func = ROOT.Math.Functor(q_integrand, 1)
        
        # Perform integration with fallback
        diff_T, status = integrate_with_fallback(func, 0.0, q_exp, q_exp, i)
        
        if status != 0:
            # Try a smaller integration range as fallback
            if q_exp > 0.01:
                smaller_range = q_exp * 0.1
                print(f"Trying smaller range {smaller_range:.6f} for q_exp={q_exp:.6f}")
                diff_T, status = integrate_with_fallback(func, 0.0, smaller_range, q_exp, i)
        
        # amplitudes
        born_amplitude = born_amp(diff_T, s, eps, t)
        eik_amplitude = eik_amp(sqrt_s, b_max, q_max, born_amplitude)  # Note: changed to sqrt_s

        # seção de choque diferencial
        diff_sigma = differential_sigma(eik_amplitude, s)
        results.append(diff_sigma)
        
        # Progress reporting
        if len(x_born) > 10 and i % max(1, len(x_born) // 10) == 0:
            print(f"Progress: {i+1}/{len(x_born)} points completed (q_exp={q_exp:.6f})")

    return np.array(results, dtype=float)

In [11]:
import ROOT
import numpy as np

# -------------------------------
# Helper to create chi² for a dataset
# -------------------------------
def make_chi2(x, y, yerr, energy, model_type):
    """
    Returns a callable that computes chi² for given model parameters
    """
    def chi2(params):
        eps, mg, a1, a2 = params
        y_pred = model_function(x, eps, mg, a1, a2, energy, model_type)
        return np.sum(((y - y_pred) / yerr) ** 2)
    return chi2

# -------------------------------
# Total chi² for ATLAS using model 'log'
# -------------------------------
def total_chi2_log_atlas(params_ptr):
    """
    params_ptr is a C++ array of 4 parameters from ROOT
    """
    eps, mg, a1, a2 = params_ptr[0], params_ptr[1], params_ptr[2], params_ptr[3]

    chi2_7  = make_chi2(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  7000,  "log")([eps, mg, a1, a2])
    chi2_8  = make_chi2(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  8000,  "log")([eps, mg, a1, a2])
    chi2_13 = make_chi2(x_13_atlas, y_13_atlas, yerr_13_atlas, 13000, "log")([eps, mg, a1, a2])

    return chi2_7 + chi2_8 + chi2_13

# -------------------------------
# Total chi² for ATLAS using model 'pl'
# -------------------------------
def total_chi2_pl_atlas(params_ptr):
    """
    params_ptr is a C++ array of 4 parameters from ROOT
    """
    eps, mg, a1, a2 = params_ptr[0], params_ptr[1], params_ptr[2], params_ptr[3]

    chi2_7  = make_chi2(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  7000,  "pl")([eps, mg, a1, a2])
    chi2_8  = make_chi2(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  8000,  "pl")([eps, mg, a1, a2])
    chi2_13 = make_chi2(x_13_atlas, y_13_atlas, yerr_13_atlas, 13000, "pl")([eps, mg, a1, a2])

    return chi2_7 + chi2_8 + chi2_13

# -------------------------------
# Create PyROOT Functors
# -------------------------------
total_cost_log_atlas = ROOT.Math.Functor(total_chi2_log_atlas, 4)  # 4 parameters: eps, mg, a1, a2
total_cost_pl_atlas  = ROOT.Math.Functor(total_chi2_pl_atlas, 4)


In [12]:
import ROOT
import numpy as np
import time

def optimization_pyroot(model_type: str, ensemble: str, initial_params: dict):
    print('\n' + '-'*80)
    print(f"Iniciando otimização dos parâmetros usando ROOT Minuit2 para {model_type} em {ensemble.upper()}")

    start_time = time.time()

    # Função de custo compatível com ROOT
    def chi2_function_for_root(x):
        # x é o ponteiro de doubles enviado pelo ROOT
        eps, mg, a1, a2 = x[0], x[1], x[2], x[3]

        # Seleção de dataset pelo ensemble
        if ensemble.lower() == 'atlas':
            x_data_sets = [x_7_atlas, x_8_atlas, x_13_atlas]
            y_data_sets = [y_7_atlas, y_8_atlas, y_13_atlas]
            yerr_data_sets = [yerr_7_atlas, yerr_8_atlas, yerr_13_atlas]
            energies = [7000, 8000, 13000]

        # elif ensemble.lower() == 'totem':
        #     x_data_sets = [x_7_totem, x_8_totem, x_13_totem]
        #     y_data_sets = [y_7_totem, y_8_totem, y_13_totem]
        #     yerr_data_sets = [yerr_7_totem, yerr_8_totem, yerr_13_totem]
        #     energies = [7000, 8000, 13000]
        
        else:
            raise ValueError("Ensemble desconhecido: use 'atlas' ou 'totem'.")

        chi2_total = 0.0
        for x_data, y_data, yerr_data, E in zip(x_data_sets, y_data_sets, yerr_data_sets, energies):
            # Usa a versão PyROOT do model_function, que integra com ROOT integrators
            y_pred = model_function(x_data, eps, mg, a1, a2, E, model_type)
            chi2_total += np.sum(((y_data - y_pred) / yerr_data) ** 2)

        return chi2_total

    # Cria o Functor ROOT
    fcn = ROOT.Math.Functor(chi2_function_for_root, 4)  # 4 parâmetros: eps, mg, a1, a2

    # Cria o minimizador ROOT
    minimizer = ROOT.Math.Factory.CreateMinimizer("Minuit2", "Migrad")
    minimizer.SetFunction(fcn)

    # Define parâmetros iniciais e limites
    eps0, mg0, a10, a20 = initial_params['eps'], initial_params['mg'], initial_params['a1'], initial_params['a2']
    minimizer.SetVariable(0, "eps", eps0, 0.01)
    minimizer.SetVariable(1, "mg",  mg0,  0.01)
    minimizer.SetVariable(2, "a1",  a10,  0.01)
    minimizer.SetVariable(3, "a2",  a20,  0.01)

    if model_type == 'pl':
        down, up = 0.80, 1.2
        minimizer.SetVariableLimits(0, down*eps0, up*eps0)
        minimizer.SetVariableLimits(1, down*mg0,  up*mg0)
        minimizer.SetVariableLimits(2, down*a10,  up*a10)
        minimizer.SetVariableLimits(3, down*a20,  up*a20)
        minimizer.SetStrategy(0)
    else:
        down, up = 0.94, 1.06
        minimizer.SetVariableLimits(1, down*mg0,  up*mg0)
        minimizer.SetVariableLimits(3, down*a20, up*a20)
        minimizer.SetStrategy(2)

    minimizer.SetTolerance(1e-2)

    # Roda minimização
    minimizer.Minimize()

    # Hesse para cálculo das incertezas
    minimizer.Hesse()

    # Extrai resultados
    x_opt = minimizer.X()
    x_err = minimizer.Errors()
    chi2_min = minimizer.MinValue()

    execution_time = time.time() - start_time
    minutes = int(execution_time // 60)
    seconds = execution_time % 60
    print(f'Tempo de execução para {model_type} em {ensemble}: {minutes} min {seconds:.2f} s \n')

    print(f"Parâmetros otimizados para {model_type} em {ensemble}:")
    print(f"eps: {x_opt[0]} ± {x_err[0]}")
    print(f"mg:  {x_opt[1]} ± {x_err[1]}")
    print(f"a1:  {x_opt[2]} ± {x_err[2]}")
    print(f"a2:  {x_opt[3]} ± {x_err[3]}")
    print(f"chi2 mínimo: {chi2_min}")

    return minimizer


In [ ]:
m_pl_atlas = optimization_pyroot(
    model_type='pl',
    ensemble='atlas',
    initial_params=initial_params_pl_atlas
)



--------------------------------------------------------------------------------
Iniciando otimização dos parâmetros usando ROOT Minuit2 para pl em ATLAS


/tmp/ipykernel_29056/3485086280.py:48: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  diff_T = np.trapz(integrand_values, q_points)
